In [23]:
# Install packages
!pip install mne pyriemann xgboost PyWavelets tensorflow scikit-learn

In [37]:
# Imports
import os
import zipfile
import numpy as np
import mne

In [38]:
# Dataset extraction
#zip_path = "Copy of eeg-motor.zip"
#extract_to = "eeg_motor_data"

#os.makedirs(extract_to, exist_ok=True)

#with zipfile.ZipFile(zip_path, 'r') as zip_ref:
#    zip_ref.extractall(extract_to)

In [39]:
# CONFIGURATION 
DATA_DIR = r"eeg_motor_data/files" 
subjects = [ 
"S001","S002","S003","S004","S005","S006","S007","S008","S009","S010",
    "S011","S012","S013","S014","S015","S016","S017","S018","S019","S020",
    "S021","S022","S023","S024","S025","S026","S027","S028","S029","S030",
    "S031","S032","S033","S034","S035","S036","S037",
    "S039","S040","S041","S042","S043","S044","S045","S046","S047","S048",
    "S049","S050","S051","S052","S053","S054","S055","S056","S057","S058",
    "S059","S060","S061","S062","S063","S064","S065","S066","S067","S068",
    "S069","S070","S071","S072","S073","S074","S075","S076","S077","S078",
    "S079","S080","S081","S082","S083","S084","S085","S086","S087",
    "S090","S091","S093","S094","S095","S096","S097","S098","S099",
    "S101","S102","S103","S105","S106","S107","S108","S109"
] 
runs=["R04","R08","R12"] 
TMIN=0 
TMAX=5 
X_all=[] 
y_all=[] 
subject_ids=[] 

In [40]:
# PROCESS EEG FILES
for subject in subjects: 
    print("="*50) 
    print("Processing:",subject) 
    print("="*50) 
    for run in runs: 
        edf_path=os.path.join( 
            DATA_DIR, 
            subject, 
            f"{subject}{run}.edf" 
        ) 
        if not os.path.exists(edf_path): 
            print("Missing:",edf_path) 
            continue 
        print("Loading:",run) 
        raw=mne.io.read_raw_edf( 
            edf_path, 
            preload=True, 
            verbose=False 
        ) 
        raw.set_eeg_reference( 
            ref_channels='average', 
            verbose=False 
        ) 
        raw.filter( 
            l_freq=0.5, 
            h_freq=50, 
            verbose=False 
        ) 
        events,event_id=mne.events_from_annotations(raw) 
        epochs=mne.Epochs( 
            raw, 
            events, 
            event_id=event_id, 
            tmin=TMIN, 
            tmax=TMAX, 
            baseline=None, 
            preload=True, 
            verbose=False 
        ) 
        X=epochs.get_data() 
        y=epochs.events[:,-1] 
        mask=np.logical_or( 
            y==2, 
            y==3 
        ) 
        X=X[mask] 
        y=y[mask] 
        y=np.where(y==2,0,1) 
        X_all.append(X) 
        y_all.append(y) 
        subject_ids.extend( 
            [subject]*len(y) 
        ) 
print("Processing Complete")

Processing: S001
Loading: R04
Used Annotations descriptions: [np.str_('T0'), np.str_('T1'), np.str_('T2')]
Loading: R08
Used Annotations descriptions: [np.str_('T0'), np.str_('T1'), np.str_('T2')]
Loading: R12
Used Annotations descriptions: [np.str_('T0'), np.str_('T1'), np.str_('T2')]
Processing: S002
Loading: R04
Used Annotations descriptions: [np.str_('T0'), np.str_('T1'), np.str_('T2')]
Loading: R08
Used Annotations descriptions: [np.str_('T0'), np.str_('T1'), np.str_('T2')]
Loading: R12
Used Annotations descriptions: [np.str_('T0'), np.str_('T1'), np.str_('T2')]
Processing: S003
Loading: R04
Used Annotations descriptions: [np.str_('T0'), np.str_('T1'), np.str_('T2')]
Loading: R08
Used Annotations descriptions: [np.str_('T0'), np.str_('T1'), np.str_('T2')]
Loading: R12
Used Annotations descriptions: [np.str_('T0'), np.str_('T1'), np.str_('T2')]
Processing: S004
Loading: R04
Used Annotations descriptions: [np.str_('T0'), np.str_('T1'), np.str_('T2')]
Loading: R08
Used Annotations de

In [31]:
# CREATE FINAL DATASET
X_all=np.concatenate(X_all,axis=0) 
y_all=np.concatenate(y_all,axis=0) 
subject_ids=np.array(subject_ids) 
print("X:",X_all.shape) 
print("Y:",y_all.shape) 
print("Subjects:",subject_ids.shape) 

X: (4326, 64, 801)
Y: (4326,)
Subjects: (4326,)


In [32]:
# SAVE DATA

save_dir=r"C:\EEG_Project\MI_numpy"

os.makedirs(
    save_dir,
    exist_ok=True
)

np.save(
    os.path.join(save_dir,
    "X_motor_imagery.npy"),
    X_all
)

np.save(
    os.path.join(save_dir,
    "y_motor_imagery.npy"),
    y_all
)

np.save(
    os.path.join(save_dir,
    "subject_ids.npy"),
    subject_ids
)

print("Saved successfully")

Saved successfully


In [42]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.model_selection import LeaveOneGroupOut
from torch.utils.data import TensorDataset, DataLoader

# ============================================================
# DEVICE
# ============================================================

device = torch.device("xpu" if torch.xpu.is_available() else "cpu")

print("Using Device:", device)

if torch.xpu.is_available():
    print(torch.xpu.get_device_name(0))

# ============================================================
# LOAD DATA
# ============================================================

X_all = np.load(r"C:\EEG_Project\MI_numpy\X_motor_imagery.npy")
y_all = np.load(r"C:\EEG_Project\MI_numpy\y_motor_imagery.npy")
subject_ids = np.load(r"C:\EEG_Project\MI_numpy\subject_ids.npy")

print("X:", X_all.shape)
print("Y:", y_all.shape)
print("Subjects:", subject_ids.shape)

# ============================================================
# SHAPE CONVERSION
# (N,64,801) -> (N,1,64,801)
# ============================================================

X_all = np.expand_dims(X_all, axis=1)

# ============================================================
# EEGNET
# ============================================================

class EEGNet(nn.Module):

    def __init__(self,
                 nb_classes=2,
                 Chans=64,
                 Samples=801):

        super().__init__()

        self.features = nn.Sequential(

            nn.Conv2d(
                1,
                8,
                kernel_size=(1,64),
                padding=(0,32),
                bias=False
            ),

            nn.BatchNorm2d(8),

            nn.Conv2d(
                8,
                16,
                kernel_size=(Chans,1),
                groups=8,
                bias=False
            ),

            nn.BatchNorm2d(16),

            nn.ELU(),

            nn.AvgPool2d(
                kernel_size=(1,4)
            ),

            nn.Dropout(0.5)
        )

        with torch.no_grad():

            dummy = torch.zeros(
                1,
                1,
                Chans,
                Samples
            )

            out = self.features(dummy)

            flatten_size = out.view(
                1,-1
            ).shape[1]

        self.classifier = nn.Linear(
            flatten_size,
            nb_classes
        )

    def forward(self,x):

        x = self.features(x)

        x = x.view(
            x.size(0),
            -1
        )

        x = self.classifier(x)

        return x

# ============================================================
# LOSO
# ============================================================

logo = LeaveOneGroupOut()

fold_scores = []

save_dir = "best_models"

os.makedirs(
    save_dir,
    exist_ok=True
)

for fold,(train_idx,test_idx) in enumerate(

    logo.split(
        X_all,
        y_all,
        groups=subject_ids
    )

):

    test_subject = np.unique(
        subject_ids[test_idx]
    )[0]

    print("\n"+"="*60)
    print(
        f"Fold {fold+1}"
        f" | Test Subject {test_subject}"
    )
    print("="*60)

    X_train = torch.tensor(
        X_all[train_idx],
        dtype=torch.float32
    )

    X_test = torch.tensor(
        X_all[test_idx],
        dtype=torch.float32
    )

    y_train = torch.tensor(
        y_all[train_idx],
        dtype=torch.long
    )

    y_test = torch.tensor(
        y_all[test_idx],
        dtype=torch.long
    )

    train_loader = DataLoader(
        TensorDataset(
            X_train,
            y_train
        ),
        batch_size=64,
        shuffle=True
    )

    test_loader = DataLoader(
        TensorDataset(
            X_test,
            y_test
        ),
        batch_size=64,
        shuffle=False
    )

    model = EEGNet(
        nb_classes=2,
        Chans=64,
        Samples=801
    ).to(device)

    criterion = nn.CrossEntropyLoss()

    optimizer = optim.Adam(
        model.parameters(),
        lr=1e-3
    )

    best_acc = 0

    EPOCHS = 25

    for epoch in range(EPOCHS):

        # =====================
        # TRAIN
        # =====================

        model.train()

        train_correct = 0
        train_total = 0

        for xb,yb in train_loader:

            xb = xb.to(device)
            yb = yb.to(device)

            optimizer.zero_grad()

            outputs = model(xb)

            loss = criterion(
                outputs,
                yb
            )

            loss.backward()

            optimizer.step()

            preds = outputs.argmax(1)

            train_correct += (
                preds == yb
            ).sum().item()

            train_total += yb.size(0)

        train_acc = (
            100 *
            train_correct /
            train_total
        )

        # =====================
        # VALIDATION
        # =====================

        model.eval()

        val_correct = 0
        val_total = 0

        with torch.no_grad():

            for xb,yb in test_loader:

                xb = xb.to(device)
                yb = yb.to(device)

                outputs = model(xb)

                preds = outputs.argmax(1)

                val_correct += (
                    preds == yb
                ).sum().item()

                val_total += yb.size(0)

        val_acc = (
            100 *
            val_correct /
            val_total
        )

        print(
            f"Epoch {epoch+1:02d}"
            f" | Train={train_acc:.2f}%"
            f" | Val={val_acc:.2f}%"
        )

        if val_acc > best_acc:

            best_acc = val_acc

            torch.save(
                model.state_dict(),
                os.path.join(
                    save_dir,
                    f"subject_{test_subject}.pth"
                )
            )

    fold_scores.append(best_acc)

    print(
        f"\nSubject {test_subject}"
        f" Best Accuracy = {best_acc:.2f}%"
    )

# ============================================================
# FINAL RESULTS
# ============================================================

print("\n")
print("="*60)
print("FINAL LOSO RESULTS")
print("="*60)

for i,acc in enumerate(fold_scores):

    print(
        f"Fold {i+1}: "
        f"{acc:.2f}%"
    )

print("="*60)

print(
    "Mean Accuracy:",
    np.mean(fold_scores)
)

print(
    "Std Accuracy:",
    np.std(fold_scores)
)

Using Device: xpu
Intel(R) Arc(TM) Graphics
X: (4326, 64, 801)
Y: (4326,)
Subjects: (4326,)

Fold 1 | Test Subject S001
Epoch 01 | Train=78.92% | Val=78.57%
Epoch 02 | Train=86.11% | Val=88.10%
Epoch 03 | Train=86.86% | Val=95.24%
Epoch 04 | Train=87.82% | Val=90.48%
Epoch 05 | Train=88.42% | Val=92.86%
Epoch 06 | Train=88.73% | Val=90.48%
Epoch 07 | Train=89.05% | Val=88.10%
Epoch 08 | Train=89.52% | Val=95.24%
Epoch 09 | Train=89.54% | Val=90.48%
Epoch 10 | Train=90.17% | Val=92.86%
Epoch 11 | Train=90.06% | Val=92.86%
Epoch 12 | Train=90.45% | Val=92.86%
Epoch 13 | Train=90.64% | Val=83.33%
Epoch 14 | Train=90.66% | Val=88.10%
Epoch 15 | Train=91.08% | Val=88.10%
Epoch 16 | Train=90.99% | Val=90.48%
Epoch 17 | Train=91.34% | Val=85.71%
Epoch 18 | Train=91.69% | Val=92.86%
Epoch 19 | Train=91.85% | Val=92.86%
Epoch 20 | Train=91.36% | Val=92.86%
Epoch 21 | Train=91.99% | Val=85.71%
Epoch 22 | Train=91.99% | Val=92.86%
Epoch 23 | Train=91.29% | Val=95.24%
Epoch 24 | Train=91.29% | Val

KeyboardInterrupt: 

In [ ]:
from sklearn.model_selection import GroupKFold
GroupKFold(n_splits=5)

In [44]:
from sklearn.model_selection import GroupKFold

gkf = GroupKFold(n_splits=5)

fold_scores = []

save_dir = "best_models_groupkfold"
os.makedirs(save_dir, exist_ok=True)

for fold, (train_idx, test_idx) in enumerate(
    gkf.split(
        X_all,
        y_all,
        groups=subject_ids
    )
):

    print("\n" + "="*60)
    print(f"Fold {fold+1}/5")
    print("="*60)

    train_subjects = np.unique(subject_ids[train_idx])
    test_subjects = np.unique(subject_ids[test_idx])

    print("Train Subjects:", len(train_subjects))
    print("Validation Subjects:", len(test_subjects))

    X_train = torch.tensor(
        X_all[train_idx],
        dtype=torch.float32
    )

    X_test = torch.tensor(
        X_all[test_idx],
        dtype=torch.float32
    )

    y_train = torch.tensor(
        y_all[train_idx],
        dtype=torch.long
    )

    y_test = torch.tensor(
        y_all[test_idx],
        dtype=torch.long
    )

    train_loader = DataLoader(
        TensorDataset(X_train, y_train),
        batch_size=64,
        shuffle=True
    )

    test_loader = DataLoader(
        TensorDataset(X_test, y_test),
        batch_size=64,
        shuffle=False
    )

    model = EEGNet(
        nb_classes=2,
        Chans=64,
        Samples=801
    ).to(device)

    criterion = nn.CrossEntropyLoss()

    optimizer = optim.Adam(
        model.parameters(),
        lr=1e-3
    )

    best_acc = 0

    EPOCHS = 25

    for epoch in range(EPOCHS):

        # ==========================
        # TRAIN
        # ==========================

        model.train()

        train_correct = 0
        train_total = 0
        train_loss = 0

        for xb, yb in train_loader:

            xb = xb.to(device)
            yb = yb.to(device)

            optimizer.zero_grad()

            outputs = model(xb)

            loss = criterion(outputs, yb)

            loss.backward()

            optimizer.step()

            train_loss += loss.item()

            preds = outputs.argmax(1)

            train_correct += (
                preds == yb
            ).sum().item()

            train_total += yb.size(0)

        train_acc = (
            100 * train_correct / train_total
        )

        # ==========================
        # VALIDATION
        # ==========================

        model.eval()

        val_correct = 0
        val_total = 0

        all_preds = []
        all_labels = []

        with torch.no_grad():

            for xb, yb in test_loader:

                xb = xb.to(device)
                yb = yb.to(device)

                outputs = model(xb)

                preds = outputs.argmax(1)

                all_preds.extend(
                    preds.cpu().numpy()
                )

                all_labels.extend(
                    yb.cpu().numpy()
                )

                val_correct += (
                    preds == yb
                ).sum().item()

                val_total += yb.size(0)

        val_acc = (
            100 * val_correct / val_total
        )

        print(
            f"Epoch {epoch+1:02d}"
            f" | Loss={train_loss/len(train_loader):.4f}"
            f" | Train={train_acc:.2f}%"
            f" | Val={val_acc:.2f}%"
        )

        if val_acc > best_acc:

            best_acc = val_acc

            torch.save(
                model.state_dict(),
                os.path.join(
                    save_dir,
                    f"fold_{fold+1}.pth"
                )
            )

    fold_scores.append(best_acc)

    print(
        f"\nFold {fold+1} Best Accuracy = "
        f"{best_acc:.2f}%"
    )

print("\n")
print("="*60)
print("GROUPKFOLD RESULTS")
print("="*60)

for i, acc in enumerate(fold_scores):

    print(
        f"Fold {i+1}: {acc:.2f}%"
    )

print("="*60)

print(
    f"Mean Accuracy: {np.mean(fold_scores):.2f}%"
)

print(
    f"Std Accuracy: {np.std(fold_scores):.2f}%"
)


Fold 1/5
Train Subjects: 82
Validation Subjects: 21
Epoch 01 | Loss=0.4776 | Train=77.26% | Val=70.75%
Epoch 02 | Loss=0.3239 | Train=85.98% | Val=81.52%
Epoch 03 | Loss=0.3052 | Train=86.90% | Val=84.47%
Epoch 04 | Loss=0.2914 | Train=87.51% | Val=84.69%
Epoch 05 | Loss=0.2758 | Train=88.97% | Val=84.81%
Epoch 06 | Loss=0.2570 | Train=88.88% | Val=85.26%
Epoch 07 | Loss=0.2462 | Train=89.81% | Val=85.71%
Epoch 08 | Loss=0.2443 | Train=89.75% | Val=84.92%
Epoch 09 | Loss=0.2357 | Train=89.72% | Val=84.47%
Epoch 10 | Loss=0.2195 | Train=90.74% | Val=81.41%
Epoch 11 | Loss=0.2201 | Train=91.23% | Val=84.24%
Epoch 12 | Loss=0.2134 | Train=91.09% | Val=84.13%
Epoch 13 | Loss=0.1959 | Train=91.52% | Val=82.65%
Epoch 14 | Loss=0.1956 | Train=91.84% | Val=82.54%
Epoch 15 | Loss=0.1985 | Train=91.41% | Val=84.58%
Epoch 16 | Loss=0.1798 | Train=92.86% | Val=84.69%
Epoch 17 | Loss=0.1878 | Train=92.31% | Val=79.59%
Epoch 18 | Loss=0.1743 | Train=92.89% | Val=81.52%
Epoch 19 | Loss=0.1726 | Trai

In [45]:
from sklearn.metrics import cohen_kappa_score

kappa = cohen_kappa_score(y_true,y_pred)
print(kappa)

NameError: name 'y_true' is not defined

In [46]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    cohen_kappa_score,
    confusion_matrix,
    classification_report
)

In [47]:
# ==========================================
# FOLD METRICS
# ==========================================

acc = accuracy_score(
    all_labels,
    all_preds
)

precision = precision_score(
    all_labels,
    all_preds,
    average='weighted'
)

recall = recall_score(
    all_labels,
    all_preds,
    average='weighted'
)

f1 = f1_score(
    all_labels,
    all_preds,
    average='weighted'
)

kappa = cohen_kappa_score(
    all_labels,
    all_preds
)

print("\n" + "="*50)
print(f"Fold {fold+1} Metrics")
print("="*50)

print(f"Accuracy      : {acc*100:.2f}%")
print(f"Precision     : {precision:.4f}")
print(f"Recall        : {recall:.4f}")
print(f"F1 Score      : {f1:.4f}")
print(f"Cohen Kappa   : {kappa:.4f}")

print("\nConfusion Matrix")
print(
    confusion_matrix(
        all_labels,
        all_preds
    )
)

print("\nClassification Report")
print(
    classification_report(
        all_labels,
        all_preds
    )
)


Fold 5 Metrics
Accuracy      : 84.88%
Precision     : 0.8518
Recall        : 0.8488
F1 Score      : 0.8485
Cohen Kappa   : 0.6976

Confusion Matrix
[[337  83]
 [ 44 376]]

Classification Report
              precision    recall  f1-score   support

           0       0.88      0.80      0.84       420
           1       0.82      0.90      0.86       420

    accuracy                           0.85       840
   macro avg       0.85      0.85      0.85       840
weighted avg       0.85      0.85      0.85       840



In [11]:
# MACHINE LEARNING IMPORTS 

import pywt 
from sklearn.model_selection import train_test_split 
from sklearn.metrics import accuracy_score 
from sklearn.metrics import classification_report 
from sklearn.metrics import confusion_matrix 
from sklearn.decomposition import PCA 
from mne.decoding import CSP 
from pyriemann.estimation import Covariances 
from pyriemann.tangentspace import TangentSpace 
from pyriemann.classification import MDM 
from xgboost import XGBClassifier

In [12]:
# LOAD DATASET 

X=np.load( 
r"C:\EEG_Project\MI_numpy\X_motor_imagery.npy" 
) 
y=np.load( 
r"C:\EEG_Project\MI_numpy\y_motor_imagery.npy" 
) 
print(X.shape) 
print(y.shape)

(4326, 64, 801)
(4326,)


In [13]:
# SPLIT DATA 

X_train,X_test,y_train,y_test=train_test_split( 
X, 
y, 
test_size=0.2, 
random_state=42, 
stratify=y 
) 

In [31]:
# FAST XGBOOST 

xgb=XGBClassifier( 
    n_estimators=100, 
    max_depth=4, 
    learning_rate=0.1, 
    subsample=0.8, 
    colsample_bytree=0.8, 
    n_jobs=-1, 
    eval_metric='logloss', 
    random_state=42 
) 
 

In [32]:
# CSP FEATURES 

csp=CSP( 
    n_components=8, 
    log=True 
) 
X_train_csp=csp.fit_transform( 
    X_train, 
    y_train 
) 
X_test_csp=csp.transform( 
    X_test 
) 
xgb.fit( 
X_train_csp, 
y_train 
) 
pred=xgb.predict( 
X_test_csp 
) 
print( 
"CSP Accuracy:", 
accuracy_score( 
y_test, 
pred 
)*100 
) 

Computing rank from data with rank=None
    Using tolerance 0.0049 (2.2e-16 eps * 64 dim * 3.5e+11  max singular value)
    Estimated rank (data): 63
    data: rank 63 computed from 64 data channels with 0 projectors
    Setting small data eigenvalues to zero (without PCA)
Reducing data rank from 64 -> 63
Estimating class=0 covariance using EMPIRICAL
Done.
Estimating class=1 covariance using EMPIRICAL
Done.
    Setting small data eigenvalues to zero (without PCA)
CSP Accuracy: 46.4203233256351


In [14]:
# EEGNET 

import tensorflow as tf 
from tensorflow.keras.models import Model 
from tensorflow.keras.layers import * 
from tensorflow.keras.constraints import max_norm 

In [15]:
import tensorflow as tf

print(tf.__version__)
print(tf.config.list_physical_devices())
print(tf.config.list_physical_devices('GPU'))

2.21.0
[PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')]
[]


In [16]:
import torch
import torch.nn as nn
import torch.optim as optim

# Device
device = torch.device("xpu" if torch.xpu.is_available() else "cpu")
print("Using:", device)

# EEGNet
class EEGNet(nn.Module):
    def __init__(self, nb_classes=2, Chans=64, Samples=801):
        super().__init__()

        self.block1 = nn.Sequential(
            nn.Conv2d(
                in_channels=1,
                out_channels=8,
                kernel_size=(1, 64),
                padding=(0, 32),
                bias=False
            ),
            nn.BatchNorm2d(8),

            nn.Conv2d(
                in_channels=8,
                out_channels=16,
                kernel_size=(Chans, 1),
                groups=8,
                bias=False
            ),
            nn.BatchNorm2d(16),
            nn.ELU(),

            nn.AvgPool2d((1, 4)),
            nn.Dropout(0.5)
        )

        # Determine flatten size automatically
        with torch.no_grad():
            dummy = torch.zeros(1, 1, Chans, Samples)
            out = self.block1(dummy)
            self.flatten_size = out.view(1, -1).shape[1]

        self.classifier = nn.Linear(
            self.flatten_size,
            nb_classes
        )

    def forward(self, x):
        x = self.block1(x)
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x

Using: xpu


In [17]:
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader
import numpy as np

X_eeg = X.reshape(
    X.shape[0],
    X.shape[1],
    X.shape[2],
    1
)

X_train, X_test, y_train, y_test = train_test_split(
    X_eeg,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# NHWC -> NCHW
X_train = np.transpose(X_train, (0, 3, 1, 2))
X_test = np.transpose(X_test, (0, 3, 1, 2))

X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)

y_train = torch.tensor(y_train, dtype=torch.long)
y_test = torch.tensor(y_test, dtype=torch.long)

train_loader = DataLoader(
    TensorDataset(X_train, y_train),
    batch_size=32,
    shuffle=True
)

test_loader = DataLoader(
    TensorDataset(X_test, y_test),
    batch_size=32,
    shuffle=False
)

In [18]:
model = EEGNet(
    nb_classes=2,
    Chans=64,
    Samples=801
).to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

In [19]:
epochs = 50

for epoch in range(epochs):

    model.train()

    running_loss = 0
    correct = 0
    total = 0

    for X_batch, y_batch in train_loader:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        outputs = model(X_batch)

        loss = criterion(outputs, y_batch)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        preds = outputs.argmax(dim=1)

        correct += (preds == y_batch).sum().item()
        total += y_batch.size(0)

    train_acc = 100 * correct / total

    print(
        f"Epoch {epoch+1}/{epochs} | "
        f"Loss: {running_loss/len(train_loader):.4f} | "
        f"Acc: {train_acc:.2f}%"
    )

Epoch 1/50 | Loss: 0.4147 | Acc: 82.23%
Epoch 2/50 | Loss: 0.3249 | Acc: 86.94%
Epoch 3/50 | Loss: 0.3075 | Acc: 87.05%
Epoch 4/50 | Loss: 0.2755 | Acc: 88.55%
Epoch 5/50 | Loss: 0.2573 | Acc: 89.39%
Epoch 6/50 | Loss: 0.2686 | Acc: 89.34%
Epoch 7/50 | Loss: 0.2614 | Acc: 89.10%
Epoch 8/50 | Loss: 0.2372 | Acc: 89.65%
Epoch 9/50 | Loss: 0.2300 | Acc: 90.49%
Epoch 10/50 | Loss: 0.2073 | Acc: 91.71%
Epoch 11/50 | Loss: 0.2120 | Acc: 90.69%
Epoch 12/50 | Loss: 0.1936 | Acc: 92.31%
Epoch 13/50 | Loss: 0.2023 | Acc: 91.73%
Epoch 14/50 | Loss: 0.1886 | Acc: 91.97%
Epoch 15/50 | Loss: 0.1796 | Acc: 92.28%
Epoch 16/50 | Loss: 0.1800 | Acc: 92.49%
Epoch 17/50 | Loss: 0.1643 | Acc: 93.01%
Epoch 18/50 | Loss: 0.1836 | Acc: 92.77%
Epoch 19/50 | Loss: 0.1799 | Acc: 92.51%
Epoch 20/50 | Loss: 0.1771 | Acc: 93.24%
Epoch 21/50 | Loss: 0.1701 | Acc: 93.32%
Epoch 22/50 | Loss: 0.1740 | Acc: 93.15%
Epoch 23/50 | Loss: 0.1564 | Acc: 93.87%
Epoch 24/50 | Loss: 0.1577 | Acc: 93.61%
Epoch 25/50 | Loss: 0.173

In [21]:
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for X_batch, y_batch in test_loader:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        outputs = model(X_batch)

        preds = outputs.argmax(dim=1)

        correct += (preds == y_batch).sum().item()
        total += y_batch.size(0)

test_acc = 100 * correct / total

print(f"\nEEGNet Test Accuracy: {test_acc:.2f}%")


EEGNet Test Accuracy: 81.87%


In [20]:
# EEGNET MODEL 

def EEGNet(nb_classes, 
           Chans=64, 
           Samples=801): 
    input_main=tf.keras.Input( 
        shape=(Chans,Samples,1) 
    ) 
    block1=Conv2D( 
        8, 
        (1,64), 
        padding='same', 
        use_bias=False 
    )(input_main) 
    block1=BatchNormalization()(block1) 
    block1=DepthwiseConv2D( 
        (Chans,1), 
        depth_multiplier=2, 
        use_bias=False, 
        depthwise_constraint=max_norm(1.) 
    )(block1) 
    block1=BatchNormalization()(block1) 
    block1=Activation('elu')(block1) 
    block1=AveragePooling2D( 
        (1,4) 
    )(block1) 
    block1=Dropout( 
        0.5 
    )(block1) 
    flatten=Flatten()(block1) 
    dense=Dense( 
        nb_classes 
    )(flatten) 
    softmax=Activation( 
        'softmax' 
    )(dense) 
    return Model( 
        inputs=input_main, 
        outputs=softmax 
    ) 

In [38]:
# PREPARE EEGNET DATA 

X_eeg=X.reshape( 
    X.shape[0], 
    X.shape[1], 
    X.shape[2], 
    1 
) 
X_train,X_test,y_train,y_test
train_test_split( 
X_eeg, 
y, 
test_size=0.2, 
random_state=42 
) 
model=EEGNet( 
nb_classes=2, 
Chans=64, 
Samples=801 
) 
model.compile( 
optimizer='adam', 
loss='sparse_categorical_crossentropy', 
metrics=['accuracy'] 
) 
history=model.fit( 
X_train, 
y_train, 
epochs=25, 
batch_size=32, 
validation_data=(X_test,y_test) 
) 
loss,acc=model.evaluate( 
X_test, 
y_test 
) 
print( 
"EEGNet Accuracy:", 
acc*100 
)

Epoch 1/25
109/109 ━━━━━━━━━━━━━━━━━━━━ 43s 349ms/step - accuracy: 0.7673 - loss: 0.5076 - val_accuracy: 0.5127 - val_loss: 0.6897
Epoch 2/25
109/109 ━━━━━━━━━━━━━━━━━━━━ 32s 293ms/step - accuracy: 0.8633 - loss: 0.3497 - val_accuracy: 0.5000 - val_loss: 0.6842
Epoch 3/25
109/109 ━━━━━━━━━━━━━━━━━━━━ 30s 273ms/step - accuracy: 0.8665 - loss: 0.3318 - val_accuracy: 0.6824 - val_loss: 0.6546
Epoch 4/25
109/109 ━━━━━━━━━━━━━━━━━━━━ 28s 255ms/step - accuracy: 0.8627 - loss: 0.3219 - val_accuracy: 0.7991 - val_loss: 0.5955
Epoch 5/25
109/109 ━━━━━━━━━━━━━━━━━━━━ 30s 272ms/step - accuracy: 0.8662 - loss: 0.3136 - val_accuracy: 0.5831 - val_loss: 0.6570
Epoch 6/25
109/109 ━━━━━━━━━━━━━━━━━━━━ 39s 249ms/step - accuracy: 0.8671 - loss: 0.3103 - val_accuracy: 0.5797 - val_loss: 1.0968
Epoch 7/25
109/109 ━━━━━━━━━━━━━━━━━━━━ 31s 285ms/step - accuracy: 0.8737 - loss: 0.3043 - val_accuracy: 0.7737 - val_loss: 0.4312
Epoch 8/25
109/109 ━━━━━━━━━━━━━━━━━━━━ 34s 317ms/step - accuracy: 0.8775 - loss: 0

## EEGNet Model Definition

Here, I define the EEGNet model architecture. This model is well-suited for EEG-based brain-computer interfaces (BCI) due to its ability to extract relevant features from raw EEG signals with relatively few parameters.

In [39]:
def EEGNet(nb_classes, Chans=64, Samples=128,
           dropoutRate=0.5, kernLength=64, F1=8, D=2, F2=16,
           dropoutType='Dropout', activation='elu'):

    if dropoutType == 'SpatialDropout2D':
        dropoutType = SpatialDropout2D
    elif dropoutType == 'Dropout':
        dropoutType = Dropout
    else:
        raise ValueError(f'dropoutType must be one of SpatialDropout2D 'f'or Dropout, got {dropoutType}')

    input_main = tf.keras.Input(shape=(Chans, Samples, 1))

    # Layer 1
    block1 = Conv2D(F1, (1, kernLength), padding='same',
                    input_shape=(Chans, Samples, 1),
                    use_bias=False)(input_main)
    block1 = BatchNormalization()(block1)
    block1 = DepthwiseConv2D((Chans, 1), use_bias=False,
                             depth_multiplier=D,
                             depthwise_constraint=max_norm(1.))(block1)
    block1 = BatchNormalization()(block1)
    block1 = Activation(activation)(block1)
    block1 = AveragePooling2D((1, 4))(block1)
    block1 = dropoutType(dropoutRate)(block1)

    # Layer 2
    block2 = SeparableConv2D(F2, (1, 16), padding='same',
                             use_bias=False)(block1)
    block2 = BatchNormalization()(block2)
    block2 = Activation(activation)(block2)
    block2 = AveragePooling2D((1, 8))(block2)
    block2 = dropoutType(dropoutRate)(block2)

    flatten = tf.keras.layers.Flatten(name='flatten')(block2)

    dense = Dense(nb_classes, name='dense',
                  kernel_constraint=max_norm(0.25))(flatten)
    softmax = Activation('softmax', name='softmax')(dense)

    return Model(inputs=input_main, outputs=softmax)

## Prepare Data for EEGNet

The EEGNet model expects input in the format `(trials, channels, samples, 1)`. The current `X_all` is `(trials, channels, samples)`. I will reshape the data to match this requirement.

In [40]:
X_eegnet = X_all.reshape(X_all.shape[0], X_all.shape[1], X_all.shape[2], 1)
y_eegnet = y_all

print("Reshaped X for EEGNet:", X_eegnet.shape)
print("Y for EEGNet:", y_eegnet.shape)

Reshaped X for EEGNet: (4326, 64, 801, 1)
Y for EEGNet: (4326,)


## Train-Test Split for EEGNet

I'll split the prepared EEGNet data into training and testing sets, similar to the previous feature extraction methods.

In [41]:
X_train_eegnet, X_test_eegnet, y_train_eegnet, y_test_eegnet = train_test_split(
    X_eegnet,
    y_eegnet,
    test_size=0.2,
    random_state=42,
    stratify=y_eegnet
)

print("EEGNet Train Shape:", X_train_eegnet.shape)
print("EEGNet Test Shape :", X_test_eegnet.shape)

EEGNet Train Shape: (3460, 64, 801, 1)
EEGNet Test Shape : (866, 64, 801, 1)


## Compile and Train EEGNet Model

Now, I will instantiate, compile, and train the EEGNet model. I'll use `Adam` optimizer and `sparse_categorical_crossentropy` as the loss function, suitable for integer-encoded labels.

In [4]:
EPOCHS = 50
BATCH_SIZE = 16
LEARNING_RATE = 0.001

# Get dimensions from the data
Chans = X_train_eegnet.shape[1]
Samples = X_train_eegnet.shape[2]
nb_classes = len(np.unique(y_train_eegnet))

eegnet_model = EEGNet(nb_classes=nb_classes, Chans=Chans, Samples=Samples,
                      dropoutRate=0.5, kernLength=64, F1=8, D=2, F2=16,
                      dropoutType='SpatialDropout2D', activation='elu')

optimizer = tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE)
eegnet_model.compile(loss='sparse_categorical_crossentropy', optimizer=optimizer, metrics=['accuracy'])

history = eegnet_model.fit(
    X_train_eegnet,
    y_train_eegnet,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    verbose=1,
    validation_data=(X_test_eegnet, y_test_eegnet)
)

NameError: name 'X_train_eegnet' is not defined

## Evaluate EEGNet Model

Finally, let's evaluate the performance of the trained EEGNet model on the test set and display the classification report and confusion matrix.

In [43]:
from sklearn.metrics import classification_report, confusion_matrix

y_pred_eegnet_prob = eegnet_model.predict(X_test_eegnet)
y_pred_eegnet = np.argmax(y_pred_eegnet_prob, axis=1)

acc_eegnet = accuracy_score(y_test_eegnet, y_pred_eegnet)

print("EEGNet Test Accuracy:", round(acc_eegnet * 100, 2), "%")

print("\n==============================")
print("EEGNet Model Report")
print("==============================")

print(
    classification_report(
        y_test_eegnet,
        y_pred_eegnet
    )
)

print("\nConfusion Matrix")

print(
    confusion_matrix(
        y_test_eegnet,
        y_pred_eegnet
    )
)

28/28 ━━━━━━━━━━━━━━━━━━━━ 4s 124ms/step
EEGNet Test Accuracy: 83.49 %

EEGNet Model Report
              precision    recall  f1-score   support

           0       0.88      0.78      0.83       433
           1       0.80      0.89      0.84       433

    accuracy                           0.83       866
   macro avg       0.84      0.83      0.83       866
weighted avg       0.84      0.83      0.83       866


Confusion Matrix
[[338  95]
 [ 48 385]]


In [44]:
import torch

print(torch.__version__)
print("XPU available:", torch.xpu.is_available() if hasattr(torch, "xpu") else False)
print("Device count:", torch.xpu.device_count() if hasattr(torch, "xpu") else 0)

2.12.1+cpu
XPU available: False
Device count: 0


In [45]:
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/xpu

Looking in indexes: https://download.pytorch.org/whl/xpu
Note: you may need to restart the kernel to use updated packages.


In [46]:
import torch

print(torch.__version__)
print(torch.xpu.is_available())
print(torch.xpu.device_count())

if torch.xpu.is_available():
    print(torch.xpu.get_device_name(0))

2.12.1+cpu
False
0


In [47]:
pip uninstall torch torchvision torchaudio -y

Found existing installation: torch 2.12.1Note: you may need to restart the kernel to use updated packages.

Uninstalling torch-2.12.1:
  Successfully uninstalled torch-2.12.1
Found existing installation: torchvision 0.27.1
Uninstalling torchvision-0.27.1:
  Successfully uninstalled torchvision-0.27.1
Found existing installation: torchaudio 2.11.0
Uninstalling torchaudio-2.11.0:
  Successfully uninstalled torchaudio-2.11.0


You can safely remove it manually.


In [1]:
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/xpu

Looking in indexes: https://download.pytorch.org/whl/xpu
  Using cached torch-2.12.1%2Bxpu-cp310-cp310-win_amd64.whl.metadata (32 kB)
  Using cached torchvision-0.27.1%2Bxpu-cp310-cp310-win_amd64.whl.metadata (5.6 kB)
  Using cached torchaudio-2.11.0%2Bxpu-cp310-cp310-win_amd64.whl.metadata (7.0 kB)
  Using cached intel_cmplr_lib_rt-2025.3.2-py2.py3-none-win_amd64.whl.metadata (1.2 kB)
  Using cached intel_cmplr_lib_ur-2025.3.2-py2.py3-none-win_amd64.whl.metadata (1.3 kB)
  Using cached intel_cmplr_lic_rt-2025.3.2-py2.py3-none-win_amd64.whl.metadata (1.2 kB)
  Using cached intel_sycl_rt-2025.3.2-py2.py3-none-win_amd64.whl.metadata (1.6 kB)
  Using cached onemkl_license-2025.3.1-py2.py3-none-win_amd64.whl.metadata (1.7 kB)
  Using cached onemkl_sycl_blas-2025.3.1-py2.py3-none-win_amd64.whl.metadata (1.8 kB)
  Using cached onemkl_sycl_dft-2025.3.1-py2.py3-none-win_amd64.whl.metadata (1.8 kB)
  Using cached onemkl_sycl_lapack-2025.3.1-py2.py3-none-win_amd64.whl.metadata (1.8 kB)
  Using c

In [1]:
import torch

print(torch.__version__)
print(torch.xpu.is_available())
print(torch.xpu.device_count())

if torch.xpu.is_available():
    print(torch.xpu.get_device_name(0))

2.12.1+xpu
True
1
Intel(R) Arc(TM) Graphics


In [2]:
import torch

device = torch.device("xpu")

x = torch.randn(5000, 5000).to(device)
y = torch.randn(5000, 5000).to(device)

z = torch.matmul(x, y)

print(z.device)

xpu:0


In [3]:
import torch

device = torch.device("xpu" if torch.xpu.is_available() else "cpu")
print(device)

xpu


In [2]:
import os
print(os.getcwd())

C:\Users\Chinnu Pc
